In [1]:
import numpy as np
import scipy.sparse as sp
import scipy.stats as stats
import matplotlib.pyplot as plt
import pandas as pd
import os
import gc
import zipfile
from numba import njit
from tqdm import tqdm

# ==========================================
# 1. ネットワークと単体複体生成 (Numba JIT)
# ==========================================
@njit
def build_simplicial_complex(N, m, seed=42):
    np.random.seed(seed)
    num_edges = 2 * m * N
    row = np.empty(num_edges, dtype=np.int32)
    col = np.empty(num_edges, dtype=np.int32)
    targets = np.empty(num_edges, dtype=np.int32)
    max_faces = m * m * N
    faces = np.empty((max_faces, 3), dtype=np.int32)

    idx, target_idx, face_idx = 0, 0, 0

    for i in range(m):
        for j in range(m):
            if i != j:
                row[idx] = i; col[idx] = j; idx += 1
                targets[target_idx] = i; target_idx += 1

    if m >= 3:
        faces[face_idx, 0] = 0; faces[face_idx, 1] = 1; faces[face_idx, 2] = 2; face_idx += 1

    chosen = np.empty(m, dtype=np.int32)

    for i in range(m, N):
        c_idx = 0
        while c_idx < m:
            r = np.random.randint(0, target_idx)
            node = targets[r]
            already = False
            for k in range(c_idx):
                if chosen[k] == node:
                    already = True; break
            if not already:
                chosen[c_idx] = node; c_idx += 1

        for j in range(m):
            for k in range(j + 1, m):
                faces[face_idx, 0] = i
                faces[face_idx, 1] = chosen[j]
                faces[face_idx, 2] = chosen[k]
                face_idx += 1

        for k in range(m):
            target = chosen[k]
            row[idx] = i; col[idx] = target; idx += 1
            row[idx] = target; col[idx] = i; idx += 1
            targets[target_idx] = i; target_idx += 1
            targets[target_idx] = target; target_idx += 1

    return row[:idx], col[:idx], faces[:face_idx]

# 異質性(Heterogeneity)を考慮した面の破裂評価
@njit
def evaluate_faces_hetero(X, faces, active_faces, theta_array):
    ruptured = 0
    for f in range(faces.shape[0]):
        if active_faces[f]:
            i, j, k = faces[f, 0], faces[f, 1], faces[f, 2]
            frustration = max(X[i], X[j], X[k]) - min(X[i], X[j], X[k])

            # 面の破裂閾値は、構成ノードの中で「最も過敏な(閾値が低い)ノード」に依存する
            face_theta = min(theta_array[i], theta_array[j], theta_array[k])

            if frustration > face_theta:
                active_faces[f] = False
                ruptured += 1
    return ruptured

# ==========================================
# 2. 異質性ロバストネスシミュレーション
# ==========================================
def run_heterogeneous_simulation(N, m, params, seed=42):
    np.random.seed(seed)
    dt = params['dt']
    T1, T2, T3, T4 = params['T1'], params['T2'], params['T3'], params['T4']

    # 異質性の導入: 各ノードに固有の閾値を正規分布で割り当て、[0.05, 0.5]でクリップ
    theta_array = np.random.normal(params['mu_theta'], params['sigma_theta'], N).astype(np.float32)
    theta_array = np.clip(theta_array, 0.05, 0.5)

    row, col, faces = build_simplicial_complex(N, m, seed)
    data = np.ones(len(row), dtype=np.float32)

    A_sparse = sp.csr_matrix((data, (row, col)), shape=(N, N))
    degrees = np.array(A_sparse.sum(axis=1)).flatten()
    degrees[degrees == 0] = 1.0
    hub_node = np.argmax(degrees)

    del row, col, data; gc.collect()

    X = np.random.uniform(0, params['epsilon'], N).astype(np.float32)
    b = np.zeros(N, dtype=np.float32)

    num_faces = faces.shape[0]
    active_faces = np.ones(num_faces, dtype=bool)

    history_betti_1 = []
    history_ks_stat = []

    total_ruptured = 0
    total_injected = 0
    X_prev = np.copy(X)

    for step in tqdm(range(T4), desc=f"Simulating Hetero N={N}", leave=False):
        if step == T1:
            b[hub_node] = 0.9 # フェイク投下
        elif step == T2:
            b[hub_node] = 0.0 # フェイク除去 (ヒステリシス)
        elif step == T3:
            # 高次元介入 (仮説3のバイパス手術)
            M_prime = int(total_ruptured * 0.95)
            total_injected += M_prime

        P = A_sparse.dot(X) / degrees
        noise = np.random.normal(0, 1, N).astype(np.float32) * np.sqrt(dt)
        dX = (P + b - X) * dt + params['sigma'] * noise
        X = np.clip(X + dX, 0, 1)

        # 異質性を考慮した面の破裂評価
        total_ruptured += evaluate_faces_hetero(X, faces, active_faces, theta_array)

        current_betti_1 = max(0, total_ruptured - total_injected)
        history_betti_1.append(current_betti_1 / max(1, num_faces))

        # Burn-in期間のKS検定
        if step <= T1 and step % 10 == 0:
            ks_stat, _ = stats.ks_2samp(X[::max(1, N//1000)], X_prev[::max(1, N//1000)])
            history_ks_stat.append((step, ks_stat))
            X_prev = np.copy(X)

    return history_betti_1, history_ks_stat, theta_array

# ==========================================
# 3. メイン実行スクリプト
# ==========================================
def main():
    N_list = [1000, 10000, 100000, 1000000]

    params = {
        'dt': 0.1,
        'T1': 100,             # Burn-in 終了
        'T2': 200,             # Pathogenesis 終了
        'T3': 300,             # Hysteresis 終了 -> 介入開始
        'T4': 400,             # 終了
        'epsilon': 0.05,
        'sigma': 0.02,
        'mu_theta': 0.25,      # 閾値の平均
        'sigma_theta': 0.08    # 閾値のばらつき (異質性)
    }

    output_dir = "Advanced_Heterogeneity_Robustness"
    os.makedirs(output_dir, exist_ok=True)
    summary_data = []

    for N in N_list:
        try:
            print(f"\n[{N:,} Nodes] ノードの異質性（Heterogeneity）ロバストネス検証開始...")
            b1_hist, ks_hist, theta_arr = run_heterogeneous_simulation(N, 3, params)

            # 1. KS検定による自己組織化証明（異質性下）
            steps_ks, ks_vals = zip(*ks_hist)
            plt.figure(figsize=(8, 5))
            plt.plot(steps_ks, ks_vals, 'g-o', linewidth=2)
            plt.title(f'Convergence to Empirical Stationary State under Heterogeneity (N={N})')
            plt.xlabel('Burn-in Steps')
            plt.ylabel('KS Statistic (Distribution Drift)')
            plt.grid(True, alpha=0.3)
            plt.savefig(os.path.join(output_dir, f'Adv_Hetero_KS_N{N}.png'), dpi=300)
            plt.close()

            # 2. 異質性分布の可視化 (N=1000の場合など代表で保存)
            if N == 100000:
                plt.figure(figsize=(8, 5))
                plt.hist(theta_arr, bins=50, color='teal', alpha=0.7, density=True)
                plt.title(f'Distribution of Heterogeneous Frustration Thresholds ($\\theta_i$)')
                plt.xlabel('Tolerance Threshold $\\theta_i$')
                plt.ylabel('Density')
                plt.grid(True, alpha=0.3)
                plt.savefig(os.path.join(output_dir, f'Adv_Hetero_Distribution_N{N}.png'), dpi=300)
                plt.close()

            # 3. 異質性下での相転移と回復プロット
            df = pd.DataFrame({'TimeStep': np.arange(params['T4']), 'Delta_Betti_1_Ratio': b1_hist})
            df.to_csv(os.path.join(output_dir, f'Adv_hetero_betti1_N{N}.csv'), index=False)

            plt.figure(figsize=(12, 6))
            plt.plot(df['TimeStep'], df['Delta_Betti_1_Ratio'], 'g-', linewidth=2.5, label='Topological Holes ($\Delta \\beta_1$)')

            plt.axvspan(0, params['T1'], color='gray', alpha=0.1, label='Phase 0: Burn-in')
            plt.axvspan(params['T1'], params['T2'], color='red', alpha=0.1, label='Phase 1a: Pathogenesis')
            plt.axvspan(params['T2'], params['T3'], color='blue', alpha=0.1, label='Phase 1b: Hysteresis')
            plt.axvspan(params['T3'], params['T4'], color='green', alpha=0.2, label='Phase 3: High-D Intervention')

            plt.title(f'Universal Phase Transition under Heterogeneous Constraints (N={N:,})')
            plt.xlabel('Time Step')
            plt.ylabel('Normalized Topological Holes ($\Delta \\beta_1 / |F|$)')
            plt.legend(loc='upper right')
            plt.grid(True, alpha=0.3)
            plt.savefig(os.path.join(output_dir, f'Adv_Hetero_PhaseTransition_N{N}.png'), dpi=300)
            plt.close()

            summary_data.append({
                'N': N,
                'Peak_Damage_Ratio': b1_hist[params['T3'] - 1],
                'Final_Betti_1_Ratio': b1_hist[-1]
            })
            gc.collect()

        except MemoryError:
            print(f"[警告] N={N} でメモリ不足。")
            break

    # 4. まとめ出力とZIP化
    if summary_data:
        df_sum = pd.DataFrame(summary_data)
        df_sum.to_csv(os.path.join(output_dir, 'Adv_hetero_summary.csv'), index=False)

        plt.figure(figsize=(8, 6))
        plt.plot(df_sum['N'], df_sum['Peak_Damage_Ratio'], 'r--o', linewidth=2.5, label='Peak Damage (Heterogeneous)')
        plt.plot(df_sum['N'], df_sum['Final_Betti_1_Ratio'], 'g-o', linewidth=2.5, label='Recovered State (Heterogeneous)')
        plt.xscale('log')
        plt.title('Robustness of High-Dimensional Recovery Across Scales')
        plt.xlabel('Network Size N (Log Scale)')
        plt.ylabel('Proportion of Topological Holes')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.savefig(os.path.join(output_dir, 'Adv_hetero_summary_plot.png'), dpi=300)
        plt.close()

        zip_filename = 'Advanced_Validation_Heterogeneity.zip'
        with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for root, _, files in os.walk(output_dir):
                for file in files:
                    zipf.write(os.path.join(root, file), arcname=file)
        print(f"\n🎉 異質性ロバストネス検証の計算完了！結果は '{zip_filename}' に出力されました。")

if __name__ == "__main__":
    main()

<>:201: SyntaxWarning: invalid escape sequence '\D'
<>:210: SyntaxWarning: invalid escape sequence '\D'
<>:201: SyntaxWarning: invalid escape sequence '\D'
<>:210: SyntaxWarning: invalid escape sequence '\D'
/tmp/ipykernel_2171/396816461.py:201: SyntaxWarning: invalid escape sequence '\D'
  plt.plot(df['TimeStep'], df['Delta_Betti_1_Ratio'], 'g-', linewidth=2.5, label='Topological Holes ($\Delta \\beta_1$)')
/tmp/ipykernel_2171/396816461.py:210: SyntaxWarning: invalid escape sequence '\D'
  plt.ylabel('Normalized Topological Holes ($\Delta \\beta_1 / |F|$)')



[1,000 Nodes] ノードの異質性（Heterogeneity）ロバストネス検証開始...



[10,000 Nodes] ノードの異質性（Heterogeneity）ロバストネス検証開始...



[100,000 Nodes] ノードの異質性（Heterogeneity）ロバストネス検証開始...



[1,000,000 Nodes] ノードの異質性（Heterogeneity）ロバストネス検証開始...



🎉 異質性ロバストネス検証の計算完了！結果は 'Advanced_Validation_Heterogeneity.zip' に出力されました。
